Installation

In [ ]:
%pip install -U -q pandas numpy openpyxl matplotlib seaborn nbformat plotly


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

Set Up

In [ ]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd()

# This supports running the notebook from either the project root or a notebooks folder.
if not (ROOT / "Dataviz_proj_all_datasets.xlsx").exists():
    ROOT = ROOT.parent

PROJECT_ROOT = ROOT
WORKBOOK = PROJECT_ROOT / "Dataviz_proj_all_datasets.xlsx"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

print("Project root:", PROJECT_ROOT)
print("Workbook exists:", WORKBOOK.exists())

Matplotlib is building the font cache; this may take a moment.


Project root: /Users/durga/Desktop/BS Course/DVD/Project/Dataviz-project-Group3
Workbook exists: True


In [3]:
sys.path.insert(0, str(SCRIPTS_DIR))

from build_dashboard_data import (
    read_workbook_tables,
    prepare_tables,
    build_analysis_tables,
)

In [4]:
raw_tables, source_label = read_workbook_tables(WORKBOOK)
tables = prepare_tables(raw_tables)

print("Loaded source:", source_label)

for name, frame in tables.items():
    print(f"{name:12} rows={len(frame):,}, columns={frame.shape[1]}")

Loaded source: workbook:Dataviz_proj_all_datasets.xlsx
customers    rows=99,441, columns=5
sellers      rows=3,095, columns=4
orders       rows=99,441, columns=8
items        rows=112,650, columns=7
reviews      rows=99,224, columns=7
products     rows=32,951, columns=10
translation  rows=71, columns=2
mql          rows=8,000, columns=4
closed       rows=842, columns=14


Products analysis

In [8]:
products = tables["products"]
items = tables["items"]

product_items = items.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

category_summary = (
    product_items
    .groupby("category", dropna=False)
    .agg(
        volume=("order_item_id", "count"),
        revenue=("price", "sum"),
        unique_products=("product_id", "nunique"),
        average_price=("price", "mean"),
    )
    .reset_index()
)

category_summary["revenue"] = category_summary["revenue"].round(2)
category_summary["average_price"] = category_summary["average_price"].round(2)

display(category_summary.head())

,category,volume,revenue,unique_products,average_price
0,agro_industry_and_commerce,212,72530.47,74,342.12
1,air_conditioning,297,55024.96,124,185.27
2,art,209,24202.64,55,115.80
3,arts_and_craftmanship,24,1814.01,19,75.58
4,audio,364,50688.50,58,139.25


In [9]:
# Categories of products with the highest volume of orders
top_by_volume = (
    category_summary
    .sort_values("volume", ascending=False)
    .head(15)
)

display(top_by_volume)

,category,volume,revenue,unique_products,average_price
7,bed_bath_table,11115,1036988.68,3029,93.30
43,health_beauty,9670,1258681.34,2444,130.16
67,sports_leisure,8641,988048.97,2867,114.34
39,furniture_decor,8334,729762.49,2657,87.56
15,computers_accessories,7827,911954.32,1639,116.51
49,housewares,6964,632248.66,2335,90.79
73,watches_gifts,5991,1205005.68,1329,201.14
70,telephony,4545,323667.53,1134,71.21
42,garden_tools,4347,485256.46,753,111.63
5,auto,4235,592720.11,1900,139.96


In [10]:
# Categories of products with the highest revenue
top_by_revenue = (
    category_summary
    .sort_values("revenue", ascending=False)
    .head(15)
)

display(top_by_revenue)

,category,volume,revenue,unique_products,average_price
43,health_beauty,9670,1258681.34,2444,130.16
73,watches_gifts,5991,1205005.68,1329,201.14
7,bed_bath_table,11115,1036988.68,3029,93.30
67,sports_leisure,8641,988048.97,2867,114.34
15,computers_accessories,7827,911954.32,1639,116.51
39,furniture_decor,8334,729762.49,2657,87.56
20,cool_stuff,3796,635290.85,789,167.36
49,housewares,6964,632248.66,2335,90.79
5,auto,4235,592720.11,1900,139.96
42,garden_tools,4347,485256.46,753,111.63


Reviews and products analysis

In [11]:
reviews = tables["reviews"]

review_by_order = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count"),
    )
)

product_items_reviews = product_items.merge(
    review_by_order,
    on="order_id",
    how="left"
)

product_items_reviews["review_group"] = pd.cut(
    product_items_reviews["review_score"],
    bins=[0, 2, 3, 5],
    labels=["Poor: 1-2", "Average: 3", "Good: 4-5"],
    include_lowest=True
)

display(product_items_reviews.head())

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,category,review_score,review_count,review_group
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,5.0,1.0,Good: 4-5
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,4.0,1.0,Good: 4-5
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,furniture_decor,5.0,1.0,Good: 4-5
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumery,4.0,1.0,Good: 4-5
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,garden_tools,5.0,1.0,Good: 4-5


In [12]:
category_order_reviews = (
    product_items[["category", "order_id"]]
    .drop_duplicates()
    .merge(
        review_by_order[["order_id", "review_score"]],
        on="order_id",
        how="left"
    )
    .dropna(subset=["review_score"])
)

category_order_reviews["review_group"] = pd.cut(
    category_order_reviews["review_score"],
    bins=[0, 2, 3, 5],
    labels=["Poor: 1-2", "Average: 3", "Good: 4-5"],
    include_lowest=True
)

display(category_order_reviews.head())

,category,order_id,review_score,review_group
0,cool_stuff,00010242fe8c5a6d1ba2dd792cb16214,5.0,Good: 4-5
1,pet_shop,00018f77f2f0320c557190d7a144bdd3,4.0,Good: 4-5
2,furniture_decor,000229ec398224ef6ca0657da4fc703e,5.0,Good: 4-5
3,perfumery,00024acbcdf0a6daa1e931b038114c75,4.0,Good: 4-5
4,garden_tools,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,Good: 4-5


In [13]:
category_review_summary = (
    category_order_reviews
    .groupby("category", dropna=False)
    .agg(
        reviewed_orders=("order_id", "nunique"),
        avg_review=("review_score", "mean"),
        poor_reviews_1_2=("review_group", lambda values: (values == "Poor: 1-2").sum()),
        average_reviews_3=("review_group", lambda values: (values == "Average: 3").sum()),
        good_reviews_4_5=("review_group", lambda values: (values == "Good: 4-5").sum()),
    )
    .reset_index()
)

category_review_summary["poor_review_rate"] = (
    category_review_summary["poor_reviews_1_2"]
    / category_review_summary["reviewed_orders"]
    * 100
)

category_review_summary["average_review_rate"] = (
    category_review_summary["average_reviews_3"]
    / category_review_summary["reviewed_orders"]
    * 100
)

category_review_summary["good_review_rate"] = (
    category_review_summary["good_reviews_4_5"]
    / category_review_summary["reviewed_orders"]
    * 100
)

category_review_summary = category_review_summary.round({
    "avg_review": 2,
    "poor_review_rate": 2,
    "average_review_rate": 2,
    "good_review_rate": 2,
})

display(
    category_review_summary
    .sort_values("avg_review")
)

,category,reviewed_orders,avg_review,poor_reviews_1_2,average_reviews_3,good_reviews_4_5,poor_review_rate,average_review_rate,good_review_rate
63,security_and_services,2,2.50,1,0,1,50.00,0.00,50.00
59,pc_gamer,8,3.12,3,0,5,37.50,0.00,62.50
62,portateis_cozinha_e_preparadores_de_alimentos,14,3.43,4,3,7,28.57,21.43,50.00
57,office_furniture,1263,3.62,286,185,792,22.64,14.65,62.71
30,fashion_male_clothing,111,3.70,29,7,75,26.13,6.31,67.57
...,...,...,...,...,...,...,...,...,...
10,books_technical,257,4.40,26,9,222,10.12,3.50,86.38
22,costruction_tools_tools,94,4.43,8,1,85,8.51,1.06,90.43
8,books_general_interest,508,4.46,42,17,449,8.27,3.35,88.39
29,fashion_childrens_clothes,8,4.50,1,0,7,12.50,0.00,87.50


In [14]:
category_sales_summary = (
    product_items
    .groupby("category", dropna=False)
    .agg(
        volume=("order_item_id", "count"),
        revenue=("price", "sum"),
        unique_products=("product_id", "nunique"),
        average_price=("price", "mean"),
    )
    .reset_index()
)

category_review_table = (
    category_sales_summary
    .merge(
        category_review_summary,
        on="category",
        how="left"
    )
)

category_review_table = category_review_table.round({
    "revenue": 2,
    "average_price": 2,
    "avg_review": 2,
})

display(
    category_review_table
    .sort_values("revenue", ascending=False)
)

,category,volume,revenue,unique_products,average_price,reviewed_orders,avg_review,poor_reviews_1_2,average_reviews_3,good_reviews_4_5,poor_review_rate,average_review_rate,good_review_rate
43,health_beauty,9670,1258681.34,2444,130.16,8771,4.18,1110,685,6976,12.66,7.81,79.53
73,watches_gifts,5991,1205005.68,1329,201.14,5576,4.07,847,479,4250,15.19,8.59,76.22
7,bed_bath_table,11115,1036988.68,3029,93.30,9313,3.97,1549,949,6815,16.63,10.19,73.18
67,sports_leisure,8641,988048.97,2867,114.34,7669,4.17,997,556,6116,13.00,7.25,79.75
15,computers_accessories,7827,911954.32,1639,116.51,6649,4.03,1064,549,5036,16.00,8.26,75.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,flowers,33,1110.04,14,33.64,28,4.39,2,2,24,7.14,7.14,85.71
46,home_comfort_2,30,760.27,5,25.34,23,3.83,5,2,16,21.74,8.70,69.57
11,cds_dvds_musicals,14,730.00,1,52.14,12,4.67,0,1,11,0.00,8.33,91.67
29,fashion_childrens_clothes,8,569.85,5,71.23,8,4.50,1,0,7,12.50,0.00,87.50


In [15]:
category_review_table["total_review_rate"] = (
    category_review_table["poor_review_rate"]
    + category_review_table["average_review_rate"]
    + category_review_table["good_review_rate"]
)

display(
    category_review_table[
        [
            "category",
            "reviewed_orders",
            "poor_review_rate",
            "average_review_rate",
            "good_review_rate",
            "total_review_rate",
        ]
    ]
    .sort_values("total_review_rate")
)

,category,reviewed_orders,poor_review_rate,average_review_rate,good_review_rate,total_review_rate
25,dvds_blu_ray,58,17.24,1.72,81.03,99.99
35,flowers,28,7.14,7.14,85.71,99.99
41,furniture_mattress_and_upholstery,38,21.05,7.89,71.05,99.99
72,unknown,1439,19.94,6.53,73.52,99.99
49,housewares,5843,13.14,8.59,78.26,99.99
...,...,...,...,...,...,...
46,home_comfort_2,23,21.74,8.70,69.57,100.01
30,fashion_male_clothing,111,26.13,6.31,67.57,100.01
8,books_general_interest,508,8.27,3.35,88.39,100.01
55,music,38,10.53,10.53,78.95,100.01


In [16]:
high_volume_threshold = category_review_table["volume"].quantile(0.75)
high_revenue_threshold = category_review_table["revenue"].quantile(0.75)

important_low_review_categories = category_review_table[
    (
        (category_review_table["volume"] >= high_volume_threshold)
        | (category_review_table["revenue"] >= high_revenue_threshold)
    )
    & (
        (category_review_table["avg_review"] < 3.5)
        | (category_review_table["poor_review_rate"] > 20)
    )
].copy()

display(
    important_low_review_categories
    .sort_values(
        ["revenue", "volume"],
        ascending=False
    )
)

,category,volume,revenue,unique_products,average_price,reviewed_orders,avg_review,poor_reviews_1_2,average_reviews_3,good_reviews_4_5,poor_review_rate,average_review_rate,good_review_rate,total_review_rate
57,office_furniture,1691,273960.7,309,162.01,1263,3.62,286,185,792,22.64,14.65,62.71,100.0


In [17]:
high_volume_threshold = category_review_table["volume"].quantile(0.5)
high_revenue_threshold = category_review_table["revenue"].quantile(0.5)

important_low_review_categories = category_review_table[
    (
        (category_review_table["volume"] >= high_volume_threshold)
        | (category_review_table["revenue"] >= high_revenue_threshold)
    )
    & (
        (category_review_table["avg_review"] < 4)
        | (category_review_table["poor_review_rate"] > 10)
    )
].copy()

display(
    important_low_review_categories
    .sort_values(
        ["revenue", "volume"],
        ascending=False
    )
)

,category,volume,revenue,unique_products,average_price,reviewed_orders,avg_review,poor_reviews_1_2,average_reviews_3,good_reviews_4_5,poor_review_rate,average_review_rate,good_review_rate,total_review_rate
43,health_beauty,9670,1258681.34,2444,130.16,8771,4.18,1110,685,6976,12.66,7.81,79.53,100.00
73,watches_gifts,5991,1205005.68,1329,201.14,5576,4.07,847,479,4250,15.19,8.59,76.22,100.00
7,bed_bath_table,11115,1036988.68,3029,93.30,9313,3.97,1549,949,6815,16.63,10.19,73.18,100.00
67,sports_leisure,8641,988048.97,2867,114.34,7669,4.17,997,556,6116,13.00,7.25,79.75,100.00
15,computers_accessories,7827,911954.32,1639,116.51,6649,4.03,1064,549,5036,16.00,8.26,75.74,100.00
39,furniture_decor,8334,729762.49,2657,87.56,6398,4.01,1064,568,4766,16.63,8.88,74.49,100.00
20,cool_stuff,3796,635290.85,789,167.36,3599,4.17,450,290,2859,12.50,8.06,79.44,100.00
49,housewares,6964,632248.66,2335,90.79,5843,4.14,768,502,4573,13.14,8.59,78.26,99.99
5,auto,4235,592720.11,1900,139.96,3877,4.09,559,305,3013,14.42,7.87,77.71,100.00
42,garden_tools,4347,485256.46,753,111.63,3496,4.14,469,279,2748,13.42,7.98,78.60,100.00


In [ ]:
# Keep categories with enough review data
bubble_data = category_review_table[
    category_review_table["reviewed_orders"] >= 11
].copy()

# Identify top categories
top_revenue = set(
    bubble_data.nlargest(10, "revenue")["category"]
)

top_volume = set(
    bubble_data.nlargest(10, "volume")["category"]
)

important_categories = top_revenue | top_volume

bubble_data["label"] = bubble_data["category"].where(
    bubble_data["category"].isin(important_categories),
    ""
)

fig = px.scatter(
    bubble_data,
    x="volume",
    y="avg_review",
    size="revenue",
    color="poor_review_rate",
    text="label",
    hover_name="category",
    hover_data={
        "volume": ":,",
        "revenue": ":,.0f",
        "avg_review": ":.2f",
        "poor_review_rate": ":.1f",
        "reviewed_orders": ":,",
        "label": False,
    },
    size_max=55,
    color_continuous_scale="RdYlGn_r",
    range_y=[1, 5],
    title=(
        "Product Categories: Sales Volume, Revenue, and Review Quality"
        "<br><sup>Categories with at least 11 reviewed orders</sup>"
    ),
    labels={
        "volume": "Items Sold",
        "avg_review": "Average Review Score",
        "revenue": "Revenue",
        "poor_review_rate": "Poor Reviews (%)",
        "reviewed_orders": "Reviewed Orders",
    },
)

# Style category labels
fig.update_traces(
    textposition="top center",
    textfont=dict(size=10),
    marker=dict(
        line=dict(width=1, color="white"),
        opacity=0.8,
    ),
)

# Add reference lines
fig.add_hline(
    y=4,
    line_dash="dash",
    line_color="gray",
    annotation_text="4.0 review threshold",
    annotation_position="bottom right",
)

fig.add_vline(
    x=bubble_data["volume"].median(),
    line_dash="dot",
    line_color="gray",
    annotation_text="Median volume",
    annotation_position="top left",
)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=750,
    title_x=0.5,
    legend_title="Poor reviews (%)",
    margin=dict(l=70, r=80, t=100, b=70),
)

# fig.show(renderer="browser")
fig.show()

In [ ]:
# Keep only categories with at least 11 reviewed orders
quadrant_data = category_review_table[
    category_review_table["reviewed_orders"] >= 11
].copy()

# Remove rows missing required values
quadrant_data = quadrant_data.dropna(
    subset=[
        "category",
        "revenue",
        "volume",
        "avg_review",
        "poor_review_rate",
    ]
).copy()

# Median thresholds
revenue_threshold = quadrant_data["revenue"].median()
volume_threshold = quadrant_data["volume"].median()

# Weak reviews:
# average review below 4 OR poor-review rate above 10%
weak_reviews = (
    (quadrant_data["avg_review"] < 4)
    | (quadrant_data["poor_review_rate"] > 10)
)

# Commercial importance:
# high revenue OR high volume
high_revenue_or_volume = (
    (quadrant_data["revenue"] >= revenue_threshold)
    | (quadrant_data["volume"] >= volume_threshold)
)

# Final risk definition:
# (high revenue OR high volume) AND weak reviews
quadrant_data["high_risk"] = (
    high_revenue_or_volume
    & weak_reviews
)

# Revenue classification
quadrant_data["revenue_level"] = np.where(
    quadrant_data["revenue"] >= revenue_threshold,
    "High revenue",
    "Low revenue",
)

# Risk classification
quadrant_data["risk_level"] = np.where(
    quadrant_data["high_risk"],
    "High risk",
    "Low risk",
)

# Four quadrants
quadrant_data["quadrant"] = (
    quadrant_data["revenue_level"]
    + " / "
    + quadrant_data["risk_level"]
)

print("Revenue threshold:", round(revenue_threshold, 2))
print("Volume threshold:", round(volume_threshold, 2))
print("Categories included:", len(quadrant_data))

Revenue threshold: 47445.71
Volume threshold: 297.0
Categories included: 71


In [21]:
display_columns = [
    "category",
    "revenue",
    "volume",
    "reviewed_orders",
    "avg_review",
    "poor_review_rate",
    "revenue_level",
    "risk_level",
    "quadrant",
]

# Sort first, then select columns
quadrant_display = (
    quadrant_data
    .sort_values(
        ["high_risk", "revenue", "volume"],
        ascending=[False, False, False],
    )[display_columns]
)

display(quadrant_display)

,category,revenue,volume,reviewed_orders,avg_review,poor_review_rate,revenue_level,risk_level,quadrant
43,health_beauty,1258681.34,9670,8771,4.18,12.66,High revenue,High risk,High revenue / High risk
73,watches_gifts,1205005.68,5991,5576,4.07,15.19,High revenue,High risk,High revenue / High risk
7,bed_bath_table,1036988.68,11115,9313,3.97,16.63,High revenue,High risk,High revenue / High risk
67,sports_leisure,988048.97,8641,7669,4.17,13.00,High revenue,High risk,High revenue / High risk
15,computers_accessories,911954.32,7827,6649,4.03,16.00,High revenue,High risk,High revenue / High risk
...,...,...,...,...,...,...,...,...,...
3,arts_and_craftmanship,1814.01,24,23,4.17,13.04,Low revenue,Low risk,Low revenue / Low risk
23,diapers_and_hygiene,1567.59,39,27,3.74,18.52,Low revenue,Low risk,Low revenue / Low risk
35,flowers,1110.04,33,28,4.39,7.14,Low revenue,Low risk,Low revenue / Low risk
46,home_comfort_2,760.27,30,23,3.83,21.74,Low revenue,Low risk,Low revenue / Low risk


In [22]:
# Assign fixed positions to the four quadrants
quadrant_data["x_base"] = np.where(
    quadrant_data["revenue_level"] == "High revenue",
    1,
    0,
)

quadrant_data["y_base"] = np.where(
    quadrant_data["risk_level"] == "High risk",
    1,
    0,
)

# Add small jitter so bubbles do not overlap completely
rng = np.random.default_rng(42)

quadrant_data["x_plot"] = (
    quadrant_data["x_base"]
    + rng.uniform(-0.20, 0.20, len(quadrant_data))
)

quadrant_data["y_plot"] = (
    quadrant_data["y_base"]
    + rng.uniform(-0.20, 0.20, len(quadrant_data))
)

quadrant_colors = {
    "High revenue / High risk": "#d73027",
    "High revenue / Low risk": "#1a9850",
    "Low revenue / High risk": "#fc8d59",
    "Low revenue / Low risk": "#91cf60",
}

fig = px.scatter(
    quadrant_data,
    x="x_plot",
    y="y_plot",
    size="volume",
    color="quadrant",
    hover_name="category",
    color_discrete_map=quadrant_colors,
    size_max=45,
    labels={
        "x_plot": "Revenue level",
        "y_plot": "Risk level",
        "quadrant": "Quadrant",
        "revenue": "Revenue",
        "volume": "Sales volume",
        "avg_review": "Average review",
        "poor_review_rate": "Poor reviews (%)",
        "reviewed_orders": "Reviewed orders",
    },
    hover_data={
        "revenue": ":,.2f",
        "volume": ":,",
        "avg_review": ":.2f",
        "poor_review_rate": ":.1f",
        "reviewed_orders": ":,",
        "revenue_level": True,
        "risk_level": True,
        "quadrant": True,
        "x_plot": False,
        "y_plot": False,
        "x_base": False,
        "y_base": False,
        "high_risk": False,
    },
    title=(
        "Product Category Risk Quadrants"
        "<br><sup>"
        "High risk = high revenue or high volume with weak reviews"
        "</sup>"
    ),
)

# Quadrant divider lines
fig.add_vline(
    x=0.5,
    line_dash="dash",
    line_color="black",
    line_width=1.5,
)

fig.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="black",
    line_width=1.5,
)

# Quadrant labels
quadrant_labels = [
    (0.15, 1.20, "LOW REVENUE<br>HIGH RISK", "#b03a2e"),
    (0.85, 1.20, "HIGH REVENUE<br>HIGH RISK", "#b03a2e"),
    (0.15, -0.20, "LOW REVENUE<br>LOW RISK", "#277a3f"),
    (0.85, -0.20, "HIGH REVENUE<br>LOW RISK", "#277a3f"),
]

for x, y, label, color in quadrant_labels:
    fig.add_annotation(
        x=x,
        y=y,
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(
            color=color,
            size=12,
        ),
    )

fig.update_xaxes(
    range=[-0.5, 1.5],
    tickmode="array",
    tickvals=[0, 1],
    ticktext=["Low revenue", "High revenue"],
    title="Revenue",
    showgrid=False,
    zeroline=False,
)

fig.update_yaxes(
    range=[-0.5, 1.5],
    tickmode="array",
    tickvals=[0, 1],
    ticktext=["Low risk", "High risk"],
    title="Review risk",
    showgrid=False,
    zeroline=False,
)

fig.update_traces(
    marker=dict(
        opacity=0.85,
        line=dict(
            width=1,
            color="white",
        ),
    )
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=750,
    hovermode="closest",
    margin=dict(
        l=80,
        r=80,
        t=150,
        b=80,
    ),
)

# fig.show(renderer="browser")
fig.show()

In [23]:
# Get categories classified as high risk by the quadrant logic
high_risk_categories = quadrant_data.loc[
    quadrant_data["high_risk"],
    "category",
].tolist()

# Rebuild joined order-level and item-level tables
orders_analysis, items_analysis = build_analysis_tables(tables)

# Keep one row per category and order
category_order_delivery = (
    items_analysis[
        ["category", "order_id"]
    ]
    .drop_duplicates()
    .merge(
        orders_analysis[
            [
                "order_id",
                "review_score",
                "is_late",
                "is_delivered",
                "delivery_days",
                "delay_days",
            ]
        ],
        on="order_id",
        how="left",
    )
)

# Keep only high-risk categories and orders with reviews
category_order_delivery = category_order_delivery[
    category_order_delivery["category"].isin(high_risk_categories)
].dropna(
    subset=["review_score"]
).copy()

# Poor review means score 1 or 2
category_order_delivery["poor_review"] = (
    category_order_delivery["review_score"] <= 2
)

# Separate poor reviews into late and non-late delivery
category_order_delivery["poor_review_delivery_group"] = np.select(
    [
        category_order_delivery["poor_review"]
        & category_order_delivery["is_late"],
        category_order_delivery["poor_review"]
        & ~category_order_delivery["is_late"],
    ],
    [
        "Poor review after late delivery",
        "Poor review without late delivery",
    ],
    default="Not a poor review",
)

display(category_order_delivery.head())

,category,order_id,review_score,is_late,is_delivered,delivery_days,delay_days,poor_review,poor_review_delivery_group
0,cool_stuff,00010242fe8c5a6d1ba2dd792cb16214,5.0,False,True,7.614421,-8.011250,False,Not a poor review
1,pet_shop,00018f77f2f0320c557190d7a144bdd3,4.0,False,True,16.216181,-2.330278,False,Not a poor review
2,furniture_decor,000229ec398224ef6ca0657da4fc703e,5.0,False,True,7.948437,-13.444954,False,Not a poor review
3,perfumery,00024acbcdf0a6daa1e931b038114c75,4.0,False,True,6.147269,-5.435660,False,Not a poor review
4,garden_tools,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,False,True,25.114352,-15.303808,False,Not a poor review


In [24]:
delivery_review_check = (
    category_order_delivery
    .groupby("category", dropna=False)
    .agg(
        reviewed_orders=("order_id", "nunique"),
        late_orders=("is_late", "sum"),
        poor_review_orders=("poor_review", "sum"),
        poor_reviews_after_late_delivery=(
            "poor_review_delivery_group",
            lambda values: (
                values == "Poor review after late delivery"
            ).sum(),
        ),
        poor_reviews_without_late_delivery=(
            "poor_review_delivery_group",
            lambda values: (
                values == "Poor review without late delivery"
            ).sum(),
        ),
        average_review=("review_score", "mean"),
        average_delay_days=("delay_days", "mean"),
    )
    .reset_index()
)

delivery_review_check["late_delivery_rate"] = (
    delivery_review_check["late_orders"]
    / delivery_review_check["reviewed_orders"]
    * 100
)

delivery_review_check["poor_review_rate"] = (
    delivery_review_check["poor_review_orders"]
    / delivery_review_check["reviewed_orders"]
    * 100
)

delivery_review_check["late_share_of_poor_reviews"] = np.where(
    delivery_review_check["poor_review_orders"] > 0,
    delivery_review_check[
        "poor_reviews_after_late_delivery"
    ]
    / delivery_review_check["poor_review_orders"]
    * 100,
    0,
)

delivery_review_check["poor_rate_after_late_delivery"] = np.where(
    delivery_review_check["late_orders"] > 0,
    delivery_review_check[
        "poor_reviews_after_late_delivery"
    ]
    / delivery_review_check["late_orders"]
    * 100,
    0,
)

on_time_orders = (
    delivery_review_check["reviewed_orders"]
    - delivery_review_check["late_orders"]
)

delivery_review_check["poor_rate_without_late_delivery"] = np.where(
    on_time_orders > 0,
    delivery_review_check[
        "poor_reviews_without_late_delivery"
    ]
    / on_time_orders
    * 100,
    0,
)

delivery_review_check = delivery_review_check.round({
    "average_review": 2,
    "average_delay_days": 2,
    "late_delivery_rate": 2,
    "poor_review_rate": 2,
    "late_share_of_poor_reviews": 2,
    "poor_rate_after_late_delivery": 2,
    "poor_rate_without_late_delivery": 2,
})

display(
    delivery_review_check.sort_values(
        "late_share_of_poor_reviews",
        ascending=False,
    )
)

,category,reviewed_orders,late_orders,poor_review_orders,poor_reviews_after_late_delivery,poor_reviews_without_late_delivery,average_review,average_delay_days,late_delivery_rate,poor_review_rate,late_share_of_poor_reviews,poor_rate_after_late_delivery,poor_rate_without_late_delivery
2,audio,347,44,76,31,45,3.83,-9.38,12.68,21.90,40.79,70.45,14.85
27,musical_instruments,622,54,82,33,49,4.18,-10.76,8.68,13.18,40.24,61.11,8.63
22,home_appliances_2,232,16,30,11,19,4.13,-11.58,6.90,12.93,36.67,68.75,8.80
34,stationery,2295,174,263,95,168,4.24,-11.35,7.58,11.46,36.12,54.60,7.92
20,health_beauty,8771,758,1110,395,715,4.18,-11.32,8.64,12.66,35.59,52.11,8.92
16,food,445,43,51,18,33,4.28,-9.17,9.66,11.46,35.29,41.86,8.21
13,electronics,2531,245,356,121,235,4.10,-10.32,9.68,14.07,33.99,49.39,10.28
10,construction_tools_lights,241,22,27,9,18,4.12,-10.79,9.13,11.20,33.33,40.91,8.22
36,toys,3853,275,488,162,326,4.19,-11.47,7.14,12.67,33.20,58.91,9.11
4,baby,2861,251,458,151,307,4.04,-10.89,8.77,16.01,32.97,60.16,11.76


In [25]:
stacked_data = (
    delivery_review_check[
        [
            "category",
            "poor_reviews_after_late_delivery",
            "poor_reviews_without_late_delivery",
            "late_share_of_poor_reviews",
            "poor_rate_after_late_delivery",
            "poor_rate_without_late_delivery",
        ]
    ]
    .sort_values(
        "late_share_of_poor_reviews",
        ascending=True,
    )
    .copy()
)

stacked_long = stacked_data.melt(
    id_vars=[
        "category",
        "late_share_of_poor_reviews",
        "poor_rate_after_late_delivery",
        "poor_rate_without_late_delivery",
    ],
    value_vars=[
        "poor_reviews_after_late_delivery",
        "poor_reviews_without_late_delivery",
    ],
    var_name="review_delivery_type",
    value_name="poor_review_orders",
)

stacked_long["review_delivery_type"] = stacked_long[
    "review_delivery_type"
].replace({
    "poor_reviews_after_late_delivery": (
        "Poor review after late delivery"
    ),
    "poor_reviews_without_late_delivery": (
        "Poor review without late delivery"
    ),
})

display(stacked_long.head())

,category,late_share_of_poor_reviews,poor_rate_after_late_delivery,poor_rate_without_late_delivery,review_delivery_type,poor_review_orders
0,market_place,13.16,38.46,12.45,Poor review after late delivery,5
1,air_conditioning,14.63,66.67,14.58,Poor review after late delivery,6
2,small_appliances,17.24,40.54,12.27,Poor review after late delivery,15
3,fixed_telephony,17.50,70.00,16.18,Poor review after late delivery,7
4,office_furniture,21.68,55.36,19.46,Poor review after late delivery,62


In [26]:
fig = px.bar(
    stacked_long,
    x="poor_review_orders",
    y="category",
    color="review_delivery_type",
    orientation="h",
    barmode="stack",
    color_discrete_map={
        "Poor review after late delivery": "#d73027",
        "Poor review without late delivery": "#f4a261",
    },
    custom_data=[
        "late_share_of_poor_reviews",
        "poor_rate_after_late_delivery",
        "poor_rate_without_late_delivery",
    ],
    labels={
        "poor_review_orders": "Number of poor reviews",
        "category": "High-risk category",
        "review_delivery_type": "Review and delivery type",
    },
    title=(
        "Poor Reviews in High-Risk Categories"
        "<br><sup>"
        "Red shows poor reviews associated with late delivery"
        "</sup>"
    ),
)

fig.update_traces(
    hovertemplate=(
        "<b>%{y}</b><br>"
        "%{fullData.name}: %{x:,}<br>"
        "Share of poor reviews after late delivery: "
        "%{customdata[0]:.1f}%<br>"
        "Poor-review rate after late delivery: "
        "%{customdata[1]:.1f}%<br>"
        "Poor-review rate without late delivery: "
        "%{customdata[2]:.1f}%"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    barmode="stack",
    width=1100,
    height=max(600, len(stacked_data) * 45),
    hovermode="closest",
    margin=dict(
        l=190,
        r=60,
        t=110,
        b=70,
    ),
)

# fig.show(renderer="browser")
fig.show()

Periodic camparsion

In [27]:
items_analysis["order_purchase_timestamp"] = pd.to_datetime(
    items_analysis["order_purchase_timestamp"],
    errors="coerce"
)

items_analysis["year"] = (
    items_analysis["order_purchase_timestamp"].dt.year
)

items_analysis["month_num"] = (
    items_analysis["order_purchase_timestamp"].dt.month
)

items_analysis["month_name"] = (
    items_analysis["order_purchase_timestamp"].dt.month_name().str[:3]
)

display(
    items_analysis["year"]
    .value_counts()
    .sort_index()
    .rename_axis("year")
    .reset_index(name="items_sold")
)

,year,items_sold
0,2016,370
1,2017,50864
2,2018,61416


In [28]:
comparison_data = items_analysis[
    items_analysis["year"].isin([2017, 2018])
    & items_analysis["month_num"].between(1, 8)
].copy()

category_year_summary = (
    comparison_data
    .groupby(["category", "year"], dropna=False)
    .agg(
        volume=("order_item_id", "count"),
        revenue=("price", "sum"),
    )
    .reset_index()
)

category_year_summary = category_year_summary.pivot(
    index="category",
    columns="year",
    values=["volume", "revenue"],
).reset_index()

category_year_summary.columns = [
    f"{metric}_{year}" if year != "" else metric
    for metric, year in category_year_summary.columns
]

category_year_summary = category_year_summary.fillna(0)

display(category_year_summary.head())

,category,volume_2017,volume_2018,revenue_2017,revenue_2018
0,agro_industry_and_commerce,22.0,151.0,4610.74,43351.27
1,air_conditioning,88.0,157.0,19214.12,25064.86
2,art,31.0,168.0,8363.95,14910.85
3,arts_and_craftmanship,2.0,22.0,151.89,1662.12
4,audio,72.0,195.0,7815.34,32960.94


In [29]:
category_growth = category_year_summary.copy()

category_growth["volume_growth_percent"] = np.where(
    category_growth["volume_2017"] > 0,
    (
        (category_growth["volume_2018"] - category_growth["volume_2017"])
        / category_growth["volume_2017"]
        * 100
    ),
    np.nan,
)

category_growth["revenue_growth_percent"] = np.where(
    category_growth["revenue_2017"] > 0,
    (
        (category_growth["revenue_2018"] - category_growth["revenue_2017"])
        / category_growth["revenue_2017"]
        * 100
    ),
    np.nan,
)

category_growth = category_growth.round({
    "volume_growth_percent": 1,
    "revenue_growth_percent": 1,
    "revenue_2017": 2,
    "revenue_2018": 2,
})

display(
    category_growth
    .sort_values("revenue_growth_percent", ascending=False)
)

,category,volume_2017,volume_2018,revenue_2017,revenue_2018,volume_growth_percent,revenue_growth_percent
66,small_appliances_home_oven_and_coffee,3.0,72.0,180.96,46623.76,2300.0,25664.7
19,construction_tools_safety,4.0,151.0,767.90,32279.72,3675.0,4103.6
18,construction_tools_lights,3.0,286.0,949.00,37026.52,9433.3,3801.6
58,party_supplies,1.0,32.0,69.90,2621.79,3100.0,3650.8
17,construction_tools_construction,30.0,795.0,3625.99,124362.31,2550.0,3329.7
...,...,...,...,...,...,...,...
11,cds_dvds_musicals,8.0,1.0,360.00,65.00,-87.5,-81.9
63,security_and_services,1.0,0.0,183.29,0.00,-100.0,-100.0
23,diapers_and_hygiene,0.0,36.0,0.00,1356.69,NaN,NaN
35,flowers,0.0,25.0,0.00,839.54,NaN,NaN


In [30]:
top_growth_categories = (
    category_growth[
        (category_growth["volume_2017"] >= 50)
        & (category_growth["volume_2018"] >= 50)
    ]
    .sort_values(
        "revenue_growth_percent",
        ascending=False
    )
    .head(20)
)

display(top_growth_categories)

,category,volume_2017,volume_2018,revenue_2017,revenue_2018,volume_growth_percent,revenue_growth_percent
45,home_appliances_2,52.0,149.0,12788.82,89252.86,186.5,597.9
37,food_drink,50.0,147.0,1295.40,7731.74,194.0,496.9
4,audio,72.0,195.0,7815.34,32960.94,170.8,321.7
44,home_appliances,184.0,528.0,14232.62,56320.70,187.0,295.7
36,food,62.0,382.0,5403.77,20575.62,516.1,280.8
68,stationery,389.0,1533.0,37414.56,136360.31,294.1,264.5
6,baby,617.0,1776.0,70788.37,256800.70,187.8,262.8
26,electronics,378.0,1863.0,29654.23,102928.59,392.9,247.1
73,watches_gifts,865.0,3703.0,210247.18,708850.94,328.1,237.2
43,health_beauty,1877.0,5951.0,247917.28,772238.15,217.0,211.5


In [31]:
monthly_category_data = (
    comparison_data
    .groupby(
        ["category", "year", "month_num", "month_name"],
        dropna=False
    )
    .agg(
        revenue=("price", "sum"),
        volume=("order_item_id", "count"),
    )
    .reset_index()
)

# Select the 12 categories with the highest combined revenue
top_categories = (
    monthly_category_data
    .groupby("category")["revenue"]
    .sum()
    .nlargest(12)
    .index
)

small_multiple_data = monthly_category_data[
    monthly_category_data["category"].isin(top_categories)
].copy()

small_multiple_data["period"] = (
    small_multiple_data["month_name"]
    + " "
    + small_multiple_data["year"].astype(str)
)

small_multiple_data = small_multiple_data.sort_values(
    ["category", "year", "month_num"]
)

display(small_multiple_data.head())

,category,year,month_num,month_name,revenue,volume,period
66,auto,2017,1,Jan,5218.53,34,Jan 2017
67,auto,2017,2,Feb,13162.40,86,Feb 2017
68,auto,2017,3,Mar,14482.07,87,Mar 2017
69,auto,2017,4,Apr,15548.17,88,Apr 2017
70,auto,2017,5,May,18640.03,125,May 2017


In [36]:
fig = px.line(
    small_multiple_data,
    x="month_num",
    y="revenue",
    color="year",
    facet_col="category",
    facet_col_wrap=3,
    markers=True,
    category_orders={
        "category": list(top_categories)
    },
    color_discrete_map={
        2017: "#087f8c",
        2018: "#d95f02",
    },
    hover_data={
        "month_name": True,
        "year": True,
        "revenue": ":,.2f",
        "volume": ":,",
        "month_num": False,
    },
    labels={
        "month_num": "Month",
        "revenue": "Revenue",
        "year": "Year",
        "category": "Category",
        "month_name": "Month",
        "volume": "Items sold",
    },
    title=(
        "Monthly Category Revenue: 2017 versus 2018"
        "<br><sup>January-August comparison for the 12 highest-revenue categories</sup>"
    ),
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(1, 9)),
    ticktext=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug"],
)

fig.update_layout(
    template="plotly_white",
    height=950,
    width=1200,
    title_x=0.5,
    hovermode="closest",
    margin=dict(
        l=60,
        r=40,
        t=110,
        b=60,
    ),
)

fig.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.replace("Category=", "")
    )
)

# fig.show(renderer="browser")
fig.show()

In [37]:
# Add order date and delivery fields to category-order records
category_month_reviews = (
    product_items[["category", "order_id"]]
    .drop_duplicates()
    .merge(
        orders_analysis[
            [
                "order_id",
                "order_purchase_timestamp",
                "review_score",
                "is_late",
                "is_delivered",
            ]
        ],
        on="order_id",
        how="left",
    )
)

category_month_reviews = category_month_reviews.dropna(
    subset=["review_score", "order_purchase_timestamp"]
).copy()

category_month_reviews["year"] = (
    category_month_reviews["order_purchase_timestamp"]
    .dt.year
)

category_month_reviews["month_num"] = (
    category_month_reviews["order_purchase_timestamp"]
    .dt.month
)

category_month_reviews["month"] = (
    category_month_reviews["order_purchase_timestamp"]
    .dt.month_name()
    .str[:3]
)

category_month_reviews["poor_review"] = (
    category_month_reviews["review_score"] <= 2
)

display(category_month_reviews.head())

,category,order_id,order_purchase_timestamp,review_score,is_late,is_delivered,year,month_num,month,poor_review
0,cool_stuff,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,5.0,False,True,2017,9,Sep,False
1,pet_shop,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,4.0,False,True,2017,4,Apr,False
2,furniture_decor,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,5.0,False,True,2018,1,Jan,False
3,perfumery,00024acbcdf0a6daa1e931b038114c75,2018-08-08 10:00:35,4.0,False,True,2018,8,Aug,False
4,garden_tools,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,5.0,False,True,2017,2,Feb,False


In [38]:
category_month_summary = (
    category_month_reviews
    .groupby(
        ["category", "year", "month_num", "month"],
        dropna=False
    )
    .agg(
        reviewed_orders=("order_id", "nunique"),
        poor_reviews=("poor_review", "sum"),
        average_review=("review_score", "mean"),
    )
    .reset_index()
)

category_month_summary["poor_review_rate"] = (
    category_month_summary["poor_reviews"]
    / category_month_summary["reviewed_orders"]
    * 100
)

category_month_summary = category_month_summary.round({
    "average_review": 2,
    "poor_review_rate": 2,
})

display(
    category_month_summary.sort_values(
        ["year", "month_num", "poor_review_rate"],
        ascending=[True, True, False],
    )
)

,category,year,month_num,month,reviewed_orders,poor_reviews,average_review,poor_review_rate
672,furniture_decor,2016,9,Sep,1,1,1.00,100.0
746,health_beauty,2016,9,Sep,1,1,1.00,100.0
1196,telephony,2016,9,Sep,1,1,1.00,100.0
183,books_technical,2016,10,Oct,1,1,1.00,100.0
510,fashion_male_clothing,2016,10,Oct,1,1,1.00,100.0
...,...,...,...,...,...,...,...,...
652,food_drink,2018,8,Aug,16,0,4.75,0.0
982,music,2018,8,Aug,6,0,4.67,0.0
1041,pc_gamer,2018,8,Aug,1,0,4.00,0.0
1106,signaling_and_security,2018,8,Aug,16,0,4.56,0.0


In [39]:
monthly_review_risk = (
    category_month_summary[
        category_month_summary["reviewed_orders"] >= 11
    ]
    .sort_values(
        "poor_review_rate",
        ascending=False,
    )
)

display(monthly_review_risk.head(30))

,category,year,month_num,month,reviewed_orders,poor_reviews,average_review,poor_review_rate
518,fashion_male_clothing,2017,8,Aug,13,7,2.85,53.85
78,audio,2018,3,Mar,33,15,2.88,45.45
21,air_conditioning,2017,2,Feb,11,5,2.73,45.45
84,auto,2016,10,Oct,11,5,2.91,45.45
253,computers_accessories,2016,10,Oct,18,8,3.06,44.44
219,christmas_supplies,2018,3,Mar,12,5,3.00,41.67
544,fashion_shoes,2018,3,Mar,12,5,3.25,41.67
709,furniture_living_room,2018,4,Apr,25,10,2.92,40.00
77,audio,2018,2,Feb,15,6,3.27,40.00
1014,office_furniture,2017,11,Nov,76,29,3.07,38.16


In [41]:
# Select the 15 categories with the most reviewed orders
top_reviewed_categories = (
    category_month_summary
    .groupby("category")["reviewed_orders"]
    .sum()
    .nlargest(15)
    .index
)

heatmap_data = category_month_summary[
    category_month_summary["category"].isin(top_reviewed_categories)
].copy()

heatmap_data["period"] = (
    heatmap_data["month"]
    + " "
    + heatmap_data["year"].astype(str)
)

review_pivot = heatmap_data.pivot_table(
    index="category",
    columns="period",
    values="poor_review_rate",
    aggfunc="first",
)

period_order = (
    heatmap_data[["period", "year", "month_num"]]
    .drop_duplicates()
    .sort_values(["year", "month_num"])["period"]
    .tolist()
)

review_pivot = review_pivot.reindex(columns=period_order)

fig = px.imshow(
    review_pivot,
    color_continuous_scale="Reds",
    aspect="auto",
    text_auto=".1f",
    labels={
        "x": "Order purchase month",
        "y": "Product category",
        "color": "Poor reviews (%)",
    },
    title=(
        "Monthly Poor-Review Rate by Product Category"
        "<br><sup>"
        "Only categories with at least 11 reviewed orders in a month"
        "</sup>"
    ),
)

fig.update_layout(
    template="plotly_white",
    width=1250,
    height=750,
    title_x=0.5,
    margin=dict(
        l=180,
        r=60,
        t=110,
        b=100,
    ),
)

# fig.show(renderer="browser")
fig.show()

In [46]:
start_date = pd.Timestamp("2017-01-01")
end_date = pd.Timestamp("2018-09-01")

orders_period = orders_analysis[
    (
        orders_analysis["order_purchase_timestamp"] >= start_date
    )
    & (
        orders_analysis["order_purchase_timestamp"] < end_date
    )
].copy()

items_period = items_analysis[
    (
        items_analysis["order_purchase_timestamp"] >= start_date
    )
    & (
        items_analysis["order_purchase_timestamp"] < end_date
    )
].copy()

print(
    "Analysis period:",
    orders_period["order_purchase_timestamp"].min(),
    "to",
    orders_period["order_purchase_timestamp"].max(),
)

print("Orders:", len(orders_period))
print("Items:", len(items_period))

Analysis period: 2017-01-05 11:56:06 to 2018-08-31 16:13:44
Orders: 99092
Items: 112279


In [48]:
category_month_reviews = (
    items_period[
        [
            "category",
            "order_id",
        ]
    ]
    .drop_duplicates()
    .merge(
        orders_period[
            [
                "order_id",
                "order_purchase_timestamp",
                "review_score",
            ]
        ],
        on="order_id",
        how="left",
    )
    .dropna(
        subset=[
            "review_score",
            "order_purchase_timestamp",
        ]
    )
    .copy()
)

category_month_reviews["year"] = (
    category_month_reviews[
        "order_purchase_timestamp"
    ].dt.year
)

category_month_reviews["month_num"] = (
    category_month_reviews[
        "order_purchase_timestamp"
    ].dt.month
)

category_month_reviews["month"] = (
    category_month_reviews[
        "order_purchase_timestamp"
    ].dt.strftime("%b")
)

category_month_reviews["period"] = (
    category_month_reviews[
        "order_purchase_timestamp"
    ].dt.to_period("M")
    .astype(str)
)

category_month_reviews["poor_review"] = (
    category_month_reviews["review_score"] <= 2
)

category_month_summary = (
    category_month_reviews
    .groupby(
        [
            "category",
            "year",
            "month_num",
            "month",
            "period",
        ],
        dropna=False,
    )
    .agg(
        reviewed_orders=("order_id", "nunique"),
        poor_reviews=("poor_review", "sum"),
        average_review=("review_score", "mean"),
    )
    .reset_index()
)

category_month_summary["poor_review_rate"] = (
    category_month_summary["poor_reviews"]
    / category_month_summary["reviewed_orders"]
    * 100
)

category_month_summary = category_month_summary.round({
    "average_review": 2,
    "poor_review_rate": 2,
})

display(
    category_month_summary.sort_values(
        [
            "period",
            "poor_review_rate",
        ],
        ascending=[True, False],
    )
)

,category,year,month_num,month,period,reviewed_orders,poor_reviews,average_review,poor_review_rate
286,construction_tools_construction,2017,1,Jan,2017-01,1,1,1.00,100.00
975,office_furniture,2017,1,Jan,2017-01,7,5,2.43,71.43
547,fashion_underwear_beach,2017,1,Jan,2017-01,2,1,3.00,50.00
704,garden_tools,2017,1,Jan,2017-01,40,14,3.42,35.00
19,air_conditioning,2017,1,Jan,2017-01,3,1,3.00,33.33
...,...,...,...,...,...,...,...,...,...
546,fashion_sport,2018,8,Aug,2018-08,1,0,3.00,0.00
633,food_drink,2018,8,Aug,2018-08,16,0,4.75,0.00
954,music,2018,8,Aug,2018-08,6,0,4.67,0.00
1012,pc_gamer,2018,8,Aug,2018-08,1,0,4.00,0.00


In [49]:
monthly_review_data = orders_period[
    orders_period["review_score"].notna()
].copy()

monthly_review_data["period"] = (
    monthly_review_data[
        "order_purchase_timestamp"
    ].dt.to_period("M")
    .astype(str)
)

monthly_review_data["poor_review"] = (
    monthly_review_data["review_score"] <= 2
)

monthly_review_summary = (
    monthly_review_data
    .groupby("period")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        poor_reviews=("poor_review", "sum"),
        average_review=("review_score", "mean"),
    )
    .reset_index()
)

monthly_review_summary["poor_review_rate"] = (
    monthly_review_summary["poor_reviews"]
    / monthly_review_summary["reviewed_orders"]
    * 100
)

monthly_delivery_data = orders_period.copy()

monthly_delivery_data["period"] = (
    monthly_delivery_data[
        "order_purchase_timestamp"
    ].dt.to_period("M")
    .astype(str)
)

monthly_delivery_summary = (
    monthly_delivery_data
    .groupby("period")
    .agg(
        total_orders=("order_id", "nunique"),
        delivered_orders=("is_delivered", "sum"),
        late_orders=("is_late", "sum"),
    )
    .reset_index()
)

monthly_delivery_summary["late_delivery_rate"] = np.where(
    monthly_delivery_summary["delivered_orders"] > 0,
    monthly_delivery_summary["late_orders"]
    / monthly_delivery_summary["delivered_orders"]
    * 100,
    np.nan,
)

monthly_summary = (
    monthly_review_summary
    .merge(
        monthly_delivery_summary,
        on="period",
        how="outer",
    )
    .sort_values("period")
)

monthly_summary = monthly_summary.round({
    "average_review": 2,
    "poor_review_rate": 2,
    "late_delivery_rate": 2,
})

display(monthly_summary)

,period,reviewed_orders,poor_reviews,average_review,poor_review_rate,total_orders,delivered_orders,late_orders,late_delivery_rate
0,2017-01,790,123,4.06,15.57,800,750,23,3.07
1,2017-02,1768,286,4.02,16.18,1780,1653,53,3.21
2,2017-03,2661,377,4.07,14.17,2682,2546,142,5.58
3,2017-04,2387,366,4.05,15.33,2404,2303,181,7.86
4,2017-05,3666,466,4.14,12.71,3700,3545,128,3.61
5,2017-06,3218,405,4.15,12.59,3245,3135,121,3.86
6,2017-07,3990,485,4.18,12.16,4026,3872,133,3.43
7,2017-08,4298,488,4.24,11.35,4331,4193,139,3.32
8,2017-09,4251,526,4.19,12.37,4285,4150,216,5.20
9,2017-10,4593,604,4.12,13.15,4631,4478,237,5.29


In [50]:
category_month_filtered = category_month_summary[
    category_month_summary["reviewed_orders"] >= 11
].copy()

top_categories = (
    category_month_filtered
    .groupby("category")["reviewed_orders"]
    .sum()
    .nlargest(15)
    .index
)

category_month_filtered = category_month_filtered[
    category_month_filtered["category"].isin(top_categories)
].copy()

display(
    category_month_filtered.sort_values(
        ["period", "poor_review_rate"],
        ascending=[True, False],
    )
)

,category,year,month_num,month,period,reviewed_orders,poor_reviews,average_review,poor_review_rate
704,garden_tools,2017,1,Jan,2017-01,40,14,3.42,35.00
82,auto,2017,1,Jan,2017-01,30,9,3.47,30.00
334,cool_stuff,2017,1,Jan,2017-01,32,6,3.88,18.75
433,electronics,2017,1,Jan,2017-01,11,2,3.82,18.18
1224,watches_gifts,2017,1,Jan,2017-01,11,2,4.00,18.18
...,...,...,...,...,...,...,...,...,...
854,housewares,2018,8,Aug,2018-08,558,57,4.32,10.22
1126,sports_leisure,2018,8,Aug,2018-08,435,41,4.38,9.43
723,garden_tools,2018,8,Aug,2018-08,122,11,4.25,9.02
265,computers_accessories,2018,8,Aug,2018-08,383,34,4.32,8.88


In [ ]:
heatmap_data = category_month_filtered.pivot(
    index="category",
    columns="period",
    values="poor_review_rate",
)

periods = pd.period_range(
    "2017-01",
    "2018-08",
    freq="M",
).astype(str)

heatmap_data = heatmap_data.reindex(
    columns=periods
)

fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Reds",
    text_auto=".1f",
    labels={
        "x": "Order month",
        "y": "Product category",
        "color": "Poor reviews (%)",
    },
    title=(
        "Monthly Poor-Review Rate by Product Category"
        "<br><sup>"
        "January 2017 to August 2018; minimum 11 reviews per category-month"
        "</sup>"
    ),
)

fig.update_layout(
    template="plotly_white",
    width=1250,
    height=750,
    title_x=0.5,
    margin=dict(
        l=190,
        r=60,
        t=110,
        b=100,
    ),
)

# fig.show(renderer="browser")
fig.show()

In [56]:
# Create the required month sequence
period_order = [
    str(period)
    for period in pd.period_range(
        "2017-01",
        "2018-08",
        freq="M",
    )
]

# Make a copy of the monthly summary
monthly_plot_data = monthly_summary.copy()

# Ensure period is a column
if "period" not in monthly_plot_data.columns:
    monthly_plot_data = (
        monthly_plot_data
        .rename_axis("period")
        .reset_index()
    )

# Keep exactly January 2017 through August 2018
monthly_plot_data = (
    monthly_plot_data
    .set_index("period")
    .reindex(period_order)
    .rename_axis("period")
    .reset_index()
)

# Convert the two measures into long format
monthly_plot_long = monthly_plot_data.melt(
    id_vars=["period"],
    value_vars=[
        "poor_review_rate",
        "late_delivery_rate",
    ],
    var_name="metric",
    value_name="rate",
)

monthly_plot_long["metric"] = monthly_plot_long["metric"].replace({
    "poor_review_rate": "Poor reviews: 1-2",
    "late_delivery_rate": "Late deliveries",
})

fig = px.line(
    monthly_plot_long,
    x="period",
    y="rate",
    color="metric",
    markers=True,
    color_discrete_map={
        "Poor reviews: 1-2": "#d73027",
        "Late deliveries": "#087f8c",
    },
    labels={
        "period": "Order purchase month",
        "rate": "Percentage (%)",
        "metric": "Measure",
    },
    title=(
        "Monthly Review and Delivery Risk"
        "<br><sup>January 2017 to August 2018</sup>"
    ),
)

fig.update_layout(
    template="plotly_white",
    width=1200,
    height=600,
    title_x=0.5,
    hovermode="x unified",
    margin=dict(
        l=70,
        r=50,
        t=110,
        b=100,
    ),
)

fig.update_xaxes(
    tickangle=45,
)

fig.update_yaxes(
    title="Percentage (%)",
)

# fig.show(renderer="browser")
fig.show()

Categories to focus

In [72]:
# Keep only January-August for both years
growth_data = items_analysis[
    items_analysis["year"].isin([2017, 2018])
    & items_analysis["month_num"].between(1, 8)
].copy()

# Sales growth by category and year
category_growth_base = (
    growth_data
    .groupby(["category", "year"], dropna=False)
    .agg(
        volume=("order_item_id", "count"),
        revenue=("price", "sum"),
    )
    .reset_index()
)

category_growth_pivot = category_growth_base.pivot(
    index="category",
    columns="year",
    values=["volume", "revenue"],
).fillna(0)

category_growth_pivot.columns = [
    f"{metric}_{year}"
    for metric, year in category_growth_pivot.columns
]

category_growth = category_growth_pivot.reset_index()

# Calculate percentage growth
category_growth["volume_growth_percent"] = np.where(
    category_growth["volume_2017"] > 0,
    (
        (category_growth["volume_2018"] - category_growth["volume_2017"])
        / category_growth["volume_2017"]
        * 100
    ),
    np.nan,
)

category_growth["revenue_growth_percent"] = np.where(
    category_growth["revenue_2017"] > 0,
    (
        (category_growth["revenue_2018"] - category_growth["revenue_2017"])
        / category_growth["revenue_2017"]
        * 100
    ),
    np.nan,
)

# Create one category-order row to avoid duplicate reviews
category_order_experience = (
    growth_data[
        ["category", "order_id"]
    ]
    .drop_duplicates()
    .merge(
        orders_analysis[
            [
                "order_id",
                "review_score",
                "is_late",
            ]
        ],
        on="order_id",
        how="left",
    )
)

# 2018 customer-experience metrics
experience_2018 = category_order_experience[
    category_order_experience["order_id"].isin(
        growth_data.loc[
            growth_data["year"] == 2018,
            "order_id",
        ]
    )
].copy()

experience_2018["poor_review"] = (
    experience_2018["review_score"] <= 2
)

category_experience_2018 = (
    experience_2018
    .dropna(subset=["review_score"])
    .groupby("category", dropna=False)
    .agg(
        reviewed_orders_2018=("order_id", "nunique"),
        avg_review_2018=("review_score", "mean"),
        poor_reviews_2018=("poor_review", "sum"),
        late_orders_2018=("is_late", "sum"),
    )
    .reset_index()
)

category_experience_2018["poor_review_rate_2018"] = (
    category_experience_2018["poor_reviews_2018"]
    / category_experience_2018["reviewed_orders_2018"]
    * 100
)

category_experience_2018["late_rate_2018"] = (
    category_experience_2018["late_orders_2018"]
    / category_experience_2018["reviewed_orders_2018"]
    * 100
)

category_focus_data = category_growth.merge(
    category_experience_2018,
    on="category",
    how="left",
)

category_focus_data = category_focus_data.round({
    "revenue_2017": 2,
    "revenue_2018": 2,
    "volume_growth_percent": 1,
    "revenue_growth_percent": 1,
    "avg_review_2018": 2,
    "poor_review_rate_2018": 2,
    "late_rate_2018": 2,
})

display(category_focus_data.head())

,category,volume_2017,volume_2018,revenue_2017,revenue_2018,volume_growth_percent,revenue_growth_percent,reviewed_orders_2018,avg_review_2018,poor_reviews_2018,late_orders_2018,poor_review_rate_2018,late_rate_2018
0,agro_industry_and_commerce,22.0,151.0,4610.74,43351.27,586.4,840.2,134.0,4.08,19.0,7.0,14.18,5.22
1,air_conditioning,88.0,157.0,19214.12,25064.86,78.4,30.5,133.0,4.08,20.0,6.0,15.04,4.51
2,art,31.0,168.0,8363.95,14910.85,441.9,78.3,164.0,3.99,27.0,15.0,16.46,9.15
3,arts_and_craftmanship,2.0,22.0,151.89,1662.12,1000.0,994.3,21.0,4.10,3.0,2.0,14.29,9.52
4,audio,72.0,195.0,7815.34,32960.94,170.8,321.7,186.0,3.85,41.0,26.0,22.04,13.98


In [58]:
# Strong customer experience criteria
strong_customer_experience = (
    (category_focus_data["reviewed_orders_2018"] >= 11)
    & (category_focus_data["avg_review_2018"] >= 4)
    & (category_focus_data["late_rate_2018"] <= 10)
)

# Growth criteria
growing_category = (
    (category_focus_data["volume_growth_percent"] > 0)
    & (category_focus_data["revenue_growth_percent"] > 0)
)

# Recommended categories combine growth and strong experience
recommended_categories = category_focus_data[
    strong_customer_experience
    & growing_category
].copy()

recommended_categories = recommended_categories.sort_values(
    [
        "revenue_growth_percent",
        "volume_growth_percent",
    ],
    ascending=False,
)

display(
    recommended_categories[
        [
            "category",
            "volume_2017",
            "volume_2018",
            "volume_growth_percent",
            "revenue_2017",
            "revenue_2018",
            "revenue_growth_percent",
            "reviewed_orders_2018",
            "avg_review_2018",
            "poor_review_rate_2018",
            "late_rate_2018",
        ]
    ]
)

,category,volume_2017,volume_2018,volume_growth_percent,revenue_2017,revenue_2018,revenue_growth_percent,reviewed_orders_2018,avg_review_2018,poor_review_rate_2018,late_rate_2018
66,small_appliances_home_oven_and_coffee,3.0,72.0,2300.0,180.96,46623.76,25664.7,71.0,4.39,8.45,7.04
18,construction_tools_lights,3.0,286.0,9433.3,949.00,37026.52,3801.6,224.0,4.15,10.27,7.59
58,party_supplies,1.0,32.0,3100.0,69.90,2621.79,3650.8,29.0,4.00,17.24,3.45
17,construction_tools_construction,30.0,795.0,2550.0,3625.99,124362.31,3329.7,639.0,4.11,13.93,8.14
13,cine_photo,5.0,64.0,1180.0,315.00,6238.46,1880.5,57.0,4.23,14.04,5.26
24,drinks,19.0,287.0,1410.5,1197.66,17424.64,1354.9,222.0,4.18,12.61,6.76
50,industry_commerce_and_business,12.0,229.0,1808.3,2356.63,30573.44,1197.3,195.0,4.19,11.79,8.72
9,books_imported,5.0,44.0,780.0,278.99,3515.86,1160.2,40.0,4.42,12.50,2.50
3,arts_and_craftmanship,2.0,22.0,1000.0,151.89,1662.12,994.3,21.0,4.10,14.29,9.52
0,agro_industry_and_commerce,22.0,151.0,586.4,4610.74,43351.27,840.2,134.0,4.08,14.18,5.22


In [69]:
focus_data = category_focus_data.copy()

# Rank growth and customer experience separately
focus_data["volume_growth_rank"] = (
    focus_data["volume_growth_percent"]
    .rank(pct=True)
)

focus_data["revenue_growth_rank"] = (
    focus_data["revenue_growth_percent"]
    .rank(pct=True)
)

focus_data["experience_rank"] = (
    focus_data["avg_review_2018"]
    .rank(pct=True)
)

# Higher score means stronger growth plus better experience
focus_data["focus_score"] = (
    focus_data["volume_growth_rank"] * 0.35
    + focus_data["revenue_growth_rank"] * 0.35
    + focus_data["experience_rank"] * 0.30
)

focus_categories = focus_data[
    strong_customer_experience
    & growing_category
].sort_values(
    "focus_score",
    ascending=False,
)

display(
    focus_categories[
        [
            "category",
            "focus_score",
            "volume_growth_percent",
            "revenue_growth_percent",
            "avg_review_2018",
            "poor_review_rate_2018",
            "late_rate_2018",
            "revenue_2018",
            "volume_2018",
        ]
    ].head(40)
)

,category,focus_score,volume_growth_percent,revenue_growth_percent,avg_review_2018,poor_review_rate_2018,late_rate_2018,revenue_2018,volume_2018
66,small_appliances_home_oven_and_coffee,0.942475,2300.0,25664.7,4.39,8.45,7.04,46623.76,72.0
22,costruction_tools_tools,0.903868,2600.0,670.9,4.44,8.22,5.48,11816.66,81.0
9,books_imported,0.875111,780.0,1160.2,4.42,12.50,2.50,3515.86,44.0
18,construction_tools_lights,0.862744,9433.3,3801.6,4.15,10.27,7.59,37026.52,286.0
13,cine_photo,0.854959,1180.0,1880.5,4.23,14.04,5.26,6238.46,64.0
50,industry_commerce_and_business,0.830301,1808.3,1197.3,4.19,11.79,8.72,30573.44,229.0
24,drinks,0.820027,1410.5,1354.9,4.18,12.61,6.76,17424.64,287.0
17,construction_tools_construction,0.812618,2550.0,3329.7,4.11,13.93,8.14,124362.31,795.0
37,food_drink,0.773230,194.0,496.9,4.40,5.88,7.56,7731.74,147.0
58,party_supplies,0.765763,3100.0,3650.8,4.00,17.24,3.45,2621.79,32.0


In [73]:
# Remove the small appliances outlier
horizontal_data = recommended_categories[
    ~recommended_categories["category"]
    .str.lower()
    .str.contains("small_appliances_home_oven_and_coffee", na=False)
].copy()

# Sort so the largest growth appears at the top
horizontal_data = horizontal_data.sort_values(
    "revenue_growth_percent",
    ascending=True
)

fig = px.bar(
    horizontal_data,
    x="revenue_growth_percent",
    y="category",
    orientation="h",
    color="avg_review_2018",
    color_continuous_scale="RdYlGn",
    range_color=[1, 5],
    hover_name="category",
    hover_data={
        "revenue_growth_percent": ":.1f",
        "volume_growth_percent": ":.1f",
        "revenue_2017": ":,.2f",
        "revenue_2018": ":,.2f",
        "volume_2017": ":,",
        "volume_2018": ":,",
        "avg_review_2018": ":.2f",
        "poor_review_rate_2018": ":.1f",
        "late_rate_2018": ":.1f",
        "reviewed_orders_2018": ":,",
    },
    labels={
        "category": "Product category",
        "revenue_growth_percent": "Revenue growth (%)",
        "volume_growth_percent": "Volume growth (%)",
        "revenue_2017": "Revenue, Jan-Aug 2017",
        "revenue_2018": "Revenue, Jan-Aug 2018",
        "volume_2017": "Volume, Jan-Aug 2017",
        "volume_2018": "Volume, Jan-Aug 2018",
        "avg_review_2018": "Average review, 2018",
        "poor_review_rate_2018": "Poor reviews, 2018 (%)",
        "late_rate_2018": "Late deliveries, 2018 (%)",
        "reviewed_orders_2018": "Reviewed orders, 2018",
    },
    title=(
        "Recommended Product Categories for Growth"
        "<br><sup>"
        "small_appliances_home_oven_and_coffee excluded as an outlier | January-August comparison"
        "</sup>"
    ),
)

fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="gray",
)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=max(600, len(horizontal_data) * 35),
    title_x=0.5,
    hovermode="closest",
    margin=dict(
        l=210,
        r=80,
        t=120,
        b=80,
    ),
)

# fig.show(renderer="browser")
fig.show()